# Tutorial 2: Model Pretraining with Masked Language Modeling

Welcome to Tutorial 2! This covers **Phase 2 & 3: Model Architecture and Training Infrastructure**.

## Learning Objectives

By the end of this tutorial, you will understand:
1. The transformer architecture for single-cell data
2. How gene and value encoders work
3. How to set up and run training with MLM
4. How to monitor training progress and save checkpoints
5. How to evaluate model performance

## Prerequisites

Complete Tutorial 1 or ensure you have preprocessed data ready.

## Part 1: Understanding the Model Architecture

scGPT-mini uses a transformer encoder architecture similar to BERT:

```
Input: Gene IDs + Expression Values
    ↓
Gene Encoder (Embedding Layer)
    +
Value Encoder (Linear Layer)
    ↓
Combined Embeddings (Gene + Value)
    ↓
Transformer Encoder (Self-Attention Layers)
    ↓
Expression Decoder → Predict masked values
Classification Decoder → Predict cell types (optional)
```

Let's build this step by step!

In [ ]:
import torch
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Part 2: Data Preparation

Let's quickly prepare data using what we learned in Tutorial 1:

In [ ]:
from scgpt_mini.data import preprocess_adata
from scgpt_mini.tokenizer import GeneVocab, tokenize_batch
from scgpt_mini.data import create_dataloader
from sklearn.model_selection import train_test_split

# Load and preprocess data
adata = sc.datasets.pbmc3k()

adata = preprocess_adata(
    adata,
    filter_gene_by_counts=10,
    filter_cell_by_genes=200,
    normalize_total_target=1e4,
    log1p=True,
    subset_hvg=500,  # Use 500 genes for faster training
    binning=False,
    inplace=False,
)

print(f"Preprocessed: {adata.n_obs} cells x {adata.n_vars} genes")

In [ ]:
# Create vocabulary
vocab = GeneVocab(adata.var_names.tolist())
print(f"Vocabulary size: {len(vocab)}")

# Tokenize
data_matrix = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
gene_names = adata.var_names.values

tokenized_data = tokenize_batch(
    data=data_matrix,
    gene_names=gene_names,
    vocab=vocab,
    append_cls=True,
    include_zero_genes=False,
    return_pt=True,
)

print(f"Tokenized {len(tokenized_data)} cells")

In [ ]:
# Split into train and validation sets
train_indices, val_indices = train_test_split(
    range(len(tokenized_data)),
    test_size=0.2,
    random_state=42,
)

train_data = [tokenized_data[i] for i in train_indices]
val_data = [tokenized_data[i] for i in val_indices]

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")

In [ ]:
# Create DataLoaders
train_loader = create_dataloader(
    tokenized_data=train_data,
    vocab=vocab,
    batch_size=32,
    max_len=1001,
    shuffle=True,
    apply_masking=True,
    mask_ratio=0.15,
)

val_loader = create_dataloader(
    tokenized_data=val_data,
    vocab=vocab,
    batch_size=32,
    max_len=1001,
    shuffle=False,
    apply_masking=True,
    mask_ratio=0.15,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## Part 3: Creating the Model

Now let's create our transformer model with scGPT-mini's default configuration:

In [ ]:
from scgpt_mini.model import TransformerModel

# Model configuration
model_config = {
    "vocab_size": len(vocab),
    "d_model": 32,          # Embedding dimension
    "nhead": 2,             # Number of attention heads
    "num_layers": 2,        # Number of transformer layers
    "d_hid": 64,            # Hidden dimension in FFN
    "dropout": 0.1,         # Dropout rate
    "max_seq_len": 1001,    # Max sequence length
    "value_mode": "continuous",  # Use continuous expression values
}

# Create model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TransformerModel(**model_config, vocab=vocab)
model = model.to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"Model created with {n_params:,} parameters")
print(f"Model size: ~{n_params * 4 / 1024 / 1024:.2f} MB")

### Understanding Model Components

Let's inspect the model architecture:

In [ ]:
print("Model Architecture:")
print(model)

print("\nKey Components:")
print(f"  Gene Encoder: {model.gene_encoder}")
print(f"  Value Encoder: {model.value_encoder}")
print(f"  Transformer: {model.num_layers} layers x {model.nhead} heads")
print(f"  Expression Decoder: {model.expr_decoder}")

### Test Forward Pass

Let's verify the model works with a single batch:

In [ ]:
# Get a sample batch
sample_batch = next(iter(train_loader))

# Move to device
genes = sample_batch['genes'].to(device)
values = sample_batch['masked_values'].to(device)
attention_mask = sample_batch['attention_mask'].to(device)

# Forward pass
model.eval()
with torch.no_grad():
    output = model(genes, values, attention_mask)

print("Forward pass output:")
for key, value in output.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape}")

## Part 4: Setting Up Training

Now we'll set up the training loop with:
- Loss function (MSE for expression prediction)
- Optimizer (AdamW)
- Learning rate scheduler (optional)
- Metrics tracking

In [ ]:
from scgpt_mini.training import Trainer
from scgpt_mini.training.losses import masked_mse_loss
from scgpt_mini.training.metrics import compute_masked_metrics

# Training configuration
training_config = {
    "learning_rate": 1e-4,
    "weight_decay": 0.01,
    "epochs": 10,
    "gradient_clip": 1.0,
    "eval_every": 1,
    "save_every": 5,
}

# Create optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=training_config['learning_rate'],
    weight_decay=training_config['weight_decay'],
)

# Optional: Learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2,
    verbose=True,
)

print("Training setup complete!")

## Part 5: Training the Model

Let's train our model! This will take 5-15 minutes on a CPU.

### Using the Trainer Class

In [ ]:
# Create output directory for checkpoints
output_dir = Path("./tutorial_output")
output_dir.mkdir(exist_ok=True)

# Create trainer
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    device=device,
    output_dir=output_dir,
    gradient_clip=training_config['gradient_clip'],
    scheduler=scheduler,
)

print("Trainer created. Starting training...")

In [ ]:
# Train the model
history = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=training_config['epochs'],
    eval_every=training_config['eval_every'],
    save_every=training_config['save_every'],
)

print("\nTraining complete!")

## Part 6: Visualizing Training Progress

Let's analyze how well the model trained:

In [ ]:
# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Training and validation loss
axes[0, 0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0, 0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('MSE Loss')
axes[0, 0].set_title('Training and Validation Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# MAE metric
axes[0, 1].plot(history['val_mae'], label='Val MAE', marker='s', color='orange')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('MAE')
axes[0, 1].set_title('Mean Absolute Error')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Pearson correlation
axes[1, 0].plot(history['val_pearson'], label='Val Pearson', marker='s', color='green')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Pearson R')
axes[1, 0].set_title('Correlation on Masked Values')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Learning rate
if 'lr' in history:
    axes[1, 1].plot(history['lr'], marker='o', color='red')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Learning Rate')
    axes[1, 1].set_title('Learning Rate Schedule')
    axes[1, 1].set_yscale('log')
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("Training curves saved!")

### Training Summary

In [ ]:
print("Training Summary:")
print(f"  Best validation loss: {min(history['val_loss']):.4f}")
print(f"  Best validation MAE: {min(history['val_mae']):.4f}")
print(f"  Best Pearson correlation: {max(history['val_pearson']):.4f}")
print(f"  Final training loss: {history['train_loss'][-1]:.4f}")
print(f"  Final validation loss: {history['val_loss'][-1]:.4f}")

# Check for overfitting
train_val_gap = history['train_loss'][-1] - history['val_loss'][-1]
if train_val_gap < -0.05:
    print("\n⚠️  Model may be underfitting (val loss < train loss)")
elif train_val_gap > 0.1:
    print("\n⚠️  Model may be overfitting (train loss << val loss)")
else:
    print("\n✅  Model is well-balanced!")

## Part 7: Evaluating Model Predictions

Let's visualize how well the model predicts masked gene expression:

In [ ]:
# Collect predictions on validation set
model.eval()
all_predictions = []
all_targets = []

with torch.no_grad():
    for batch in val_loader:
        genes = batch['genes'].to(device)
        values = batch['masked_values'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        mask_positions = batch['mask_positions'].to(device)
        targets = batch['target_values'].to(device)
        
        # Forward pass
        output = model(genes, values, attention_mask)
        predictions = output['expr_pred']
        
        # Extract masked positions
        masked_preds = predictions[mask_positions]
        masked_targets = targets[mask_positions]
        
        all_predictions.append(masked_preds.cpu())
        all_targets.append(masked_targets.cpu())

# Concatenate
all_predictions = torch.cat(all_predictions).numpy()
all_targets = torch.cat(all_targets).numpy()

print(f"Collected {len(all_predictions):,} masked predictions")

In [ ]:
# Scatter plot: Predicted vs True
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Sample points for visualization (too many to plot)
n_sample = min(10000, len(all_predictions))
indices = np.random.choice(len(all_predictions), n_sample, replace=False)

# Scatter plot
axes[0].scatter(
    all_targets[indices],
    all_predictions[indices],
    alpha=0.3,
    s=1,
)
axes[0].plot(
    [all_targets.min(), all_targets.max()],
    [all_targets.min(), all_targets.max()],
    'r--',
    label='Perfect prediction',
)
axes[0].set_xlabel('True Expression')
axes[0].set_ylabel('Predicted Expression')
axes[0].set_title('Predicted vs True Expression')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residuals
residuals = all_predictions - all_targets
axes[1].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='r', linestyle='--', label='Zero error')
axes[1].set_xlabel('Prediction Error')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Prediction Errors')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'predictions.png', dpi=150, bbox_inches='tight')
plt.show()

# Compute correlation
from scipy.stats import pearsonr
r, p = pearsonr(all_targets, all_predictions)
print(f"\nPearson correlation: r = {r:.4f} (p < {p:.2e})")
print(f"Mean absolute error: {np.abs(residuals).mean():.4f}")
print(f"Root mean squared error: {np.sqrt((residuals**2).mean()):.4f}")

## Part 8: Saving and Loading Checkpoints

Always save your trained models!

In [ ]:
# Save final checkpoint
checkpoint_path = output_dir / "final_model.pt"
trainer.save_checkpoint(
    checkpoint_path,
    epoch=training_config['epochs'],
    is_best=True,
)

print(f"Model saved to: {checkpoint_path}")
print(f"Checkpoint size: {checkpoint_path.stat().st_size / 1024 / 1024:.2f} MB")

In [ ]:
# Test loading the checkpoint
new_model = TransformerModel(**model_config, vocab=vocab)
checkpoint = torch.load(checkpoint_path, map_location=device)
new_model.load_state_dict(checkpoint['model_state_dict'])
new_model = new_model.to(device)

print("✅ Checkpoint loaded successfully!")
print(f"   Trained for {checkpoint['epoch']} epochs")
print(f"   Best validation loss: {checkpoint.get('best_val_loss', 'N/A')}")

## Summary

In this tutorial, you learned:

✅ **Model Architecture**:
- Gene and value encoders
- Transformer encoder layers
- Expression decoder

✅ **Training Setup**:
- Loss functions (MSE for MLM)
- Optimizers and schedulers
- Training loops and metrics

✅ **Model Training**:
- Masked language modeling objective
- Monitoring training progress
- Detecting overfitting

✅ **Evaluation**:
- Computing metrics (MSE, MAE, Pearson)
- Visualizing predictions
- Analyzing errors

✅ **Checkpointing**:
- Saving trained models
- Loading for inference

## Next Steps

Continue to **Tutorial 3: Cell Embeddings** to learn how to extract meaningful representations from your trained model!

## Exercises

Try these to deepen your understanding:

1. **Larger model**: Increase d_model to 64 and num_layers to 3. How does performance change?
2. **Different mask ratios**: Try 5%, 30%, 50% masking. Which works best?
3. **Longer training**: Train for 50 epochs. Does performance keep improving?
4. **Binned values**: Use `value_mode="binned"` instead of continuous
5. **Custom data**: Train on your own single-cell dataset
6. **Attention visualization**: Extract attention weights and visualize gene-gene relationships
7. **Early stopping**: Implement early stopping if validation loss doesn't improve